In [ ]:
# DFU Phase-5 Paper Evidence V2 — verified 8-part loader; no training/inference
import urllib.request, hashlib, base64, zlib

VERSION = "DFU_PHASE5_PAPER_EVIDENCE_LOADER_V2_20260812"
SOURCE_COMMIT = "e8d38607767880ce50c4460cc7774c528845ad6d"
PARTS = [('scripts/phase5_paper_evidence_v2_payload/part_00.txt', '690c03d37e07460ebef0bb655bfb937917869a56'), ('scripts/phase5_paper_evidence_v2_payload/part_01.txt', 'd4779c540f0223a1aed9673a06f7ed2987f9f3bb'), ('scripts/phase5_paper_evidence_v2_payload/part_02.txt', '65c40c7b5178d362dca4641d29ef81a2a6a4003c'), ('scripts/phase5_paper_evidence_v2_payload/part_03.txt', '5afb995dda948b9b8f839f4600f10dd8384903af'), ('scripts/phase5_paper_evidence_v2_payload/part_04.txt', 'a2a99b42e907feb7dcaa76773346eea2cbe41fdb'), ('scripts/phase5_paper_evidence_v2_payload/part_05.txt', '6dfebf918540e840a9bb7d97a1330c817c4d9fea'), ('scripts/phase5_paper_evidence_v2_payload/part_06.txt', '8db75a19ac5cddbb120e16e9aa9f5ebf3b6dc87b'), ('scripts/phase5_paper_evidence_v2_payload/part_07.txt', '96717a8e2d1278b04e6834f88d20dbc5ef50e08c')]
EXPECTED_PAYLOAD_SHA256 = "4cd29975969b670db050a498833e81329d71ae899323099534cfd5edf41cd0ab"
EXPECTED_SOURCE_SHA256 = "7052d18cbbb3821d7b704fb68e1b94e863ec4ed4b0302214c311032231c077bd"
BASE = f"https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/{SOURCE_COMMIT}/"

def git_blob_sha(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

print("="*100)
print(VERSION)
print("POST-HOC PACKAGING ONLY | NO TRAINING | NO CNN INFERENCE | DRIVE ARTIFACTS ONLY")
print("="*100)
chunks=[]
for path, expected_blob in PARTS:
    raw=urllib.request.urlopen(BASE+path, timeout=120).read()
    actual_blob=git_blob_sha(raw)
    if actual_blob != expected_blob:
        raise RuntimeError(f"Payload fragment mismatch: {path} expected={expected_blob} actual={actual_blob}")
    print("Payload fragment PASS:", path, actual_blob, "bytes=", len(raw))
    chunks.append(raw)
payload=b"".join(chunks)
psha=hashlib.sha256(payload).hexdigest()
if psha != EXPECTED_PAYLOAD_SHA256:
    raise RuntimeError(f"Payload SHA mismatch: expected={EXPECTED_PAYLOAD_SHA256} actual={psha}")
print("Combined payload SHA256: PASS", psha)
source=zlib.decompress(base64.b64decode(payload, validate=True))
ssha=hashlib.sha256(source).hexdigest()
if ssha != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(f"Decoded source SHA mismatch: expected={EXPECTED_SOURCE_SHA256} actual={ssha}")
print("Decoded source SHA256: PASS", ssha)
text=source.decode("utf-8")
compile(text, "dfu_phase5_paper_evidence_v2.py", "exec")
print("Decoded source compile: PASS")
print("Starting Phase-5 packaging...")
exec(compile(text, "dfu_phase5_paper_evidence_v2.py", "exec"), globals())
